# LENO for the KPP-Fisher Equation

Learned Evolution Neural Operator (LENO) for the 1D KPP-Fisher reaction–diffusion equation.
The model learns a nonlinear operator in a truncated sine basis and couples it with an
implicit linear diffusion step via spectral eigenvalues.

## Imports

In [29]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from timeit import default_timer as timer

## Data Loading and Hyperparameters

Load FEM-generated trajectories from `data_kpp.npy` and set the time step, basis size,
network width, and number of training samples.

In [ ]:
# device = 'cuda:0'
device = 'cpu'
dt = 1/100
sol = np.load('./data_gen/data/data_kpp.npy') # FEM-generated trajectories: 100 x 101 x 1025, (samples x time steps x grid points)  
num_steps = 20
num_points = sol.shape[-1] 
x = np.linspace(0, 1, num_points)
p = 64 # number of eigenfunctions  
M = 1000 # number of hidden neurons 
ntrain = 100 # number of training samples 

In [37]:
print(sol.shape)
print(x.shape)  

torch.Size([100, 101, 1025])
(1025,)


## Spectral Coefficient Extraction

Project the solution onto the first $p$ sine modes and compute the modal coefficients
$\beta$ by averaging over the spatial grid. Prepare PyTorch tensors for training.

eigenfunctions: $$\phi_i = \sqrt{2} \sin( k \pi x), k = 1,2,3..., (\phi_i,\phi_j) =\delta_{i,j}$$, 

\begin{equation}

\beta_k(t) = \int_0^1 u(x,t) \phi_k(x)\,dx \approx

\frac{\Delta x}{2}

\sum_{j=0}^{N_x-2}

\Bigl( u(x_j,t)\phi_k(x_j) +
u(x_{j+1},t)\phi_k(x_{j+1}) ) \Bigr).

\end{equation}

In [60]:
# beta = np.zeros([sol.shape[0], sol.shape[1], p]) # (samples x time steps x eigen coefficients)  

# dx = x[1] - x[0] 
# for i in range(p):
#     basis = sol * np.sin((i + 1) * np.pi * x)
#     # beta[:, :, i] = 1/2 * dx * np.sum(basis[..., :-1] + basis[..., 1:], -1)
#     beta[:, :, i] = np.mean(basis[..., :-1] + basis[..., 1:], -1)

sqrt2 = np.sqrt(2.0)
sqrt2_torch = torch.sqrt(torch.tensor(2.0, dtype=torch.double, device=device))  
beta = np.zeros([sol.shape[0], sol.shape[1], p])

dx = x[1] - x[0]

for i in range(p):
    phi = sqrt2 * np.sin((i + 1) * np.pi * x) 
    basis = sol * phi
    beta[:, :, i] = 0.5 * dx * np.sum(basis[..., :-1] + basis[..., 1:], axis=-1)
target = beta # (samples x time steps x eigen coefficients)  

sol_torch = torch.tensor(sol).to(device) 
myall = torch.tensor(target[:ntrain, :, :]).to(device)
initial = torch.tensor(target[:ntrain, 0, :]).to(device)
target = torch.tensor(target[:ntrain, 1:, :]).to(device) 
eigen = (torch.linspace(1, p, p) * torch.pi).to(device) ** 2

In [61]:
sol_torch.size()

torch.Size([100, 101, 1025])

## Neural Network Architecture

 $$\sum_{i = 1}^P \mathcal{G}_i(\beta(u);\theta)\phi_i(x)$$

\begin{equation}
        \frac{\tilde{\beta}^n - \tilde{\beta}^{n-1}}{t_n - t_{n-1}} + \Lambda_P \tilde{\beta}^n = \mathcal{G}(\tilde{\beta}^{n-1};\theta),
\end{equation}

A fully connected network maps the $p$-dimensional spectral coefficients to the learned
nonlinear term at each time step.



In [58]:
class NonlinearNet(nn.Module):
    def __init__(self, M, p):
        super(NonlinearNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(p, M),
            nn.ReLU(),
            nn.Linear(M, M),
            nn.ReLU(),
            nn.Linear(M, p)
        )

    def forward(self, u):
        return self.net(u)

## Training Setup

Initialize the network, Adam optimizer, and step learning-rate scheduler.
Each epoch unrolls `num_steps` semi-implicit spectral updates and minimizes:
1. **Trajectory loss** — relative $L^2$ error on predicted vs. target coefficients.
2. **Residual loss** — consistency of the learned nonlinear term with finite-difference dynamics.

In [ ]:
epochs = 500 #5000
lr = 1e-3
net = NonlinearNet(M, p).to(device)
net = net.double()
optimizer = optim.Adam(net.parameters(), lr=lr)
step_size = 100 # 
gamma = 0.25
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
train_err = []
t1 = 0

for i in range(epochs):
    u0 = initial
    loss = 0
    for step in range(num_steps):
        u = (u0 + dt * net(u0)) / (1 + dt * eigen)
        loss += torch.mean(torch.norm(u - target[:, step, :], 2, -1) / torch.norm(target[:, step, :], 2, -1)) / num_steps
        Non = net(myall[:, step, :])
        ff = (myall[:, step + 1, :] - myall[:, step, :]) / dt + myall[:, step + 1, :] * eigen
        loss += torch.mean(torch.norm(ff - Non, 2, -1) / torch.norm(ff, 2, -1)) / num_steps 
        u0 = u
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    train_err.append(loss.item())
    if (i + 1) % 50 == 0:
        t2 = timer()
        print((t2 - t1) / 50)
        t1 = t2
        print(f"Step {i + 1}, Loss: {loss.item()}")

764.08831329832
Step 50, Loss: 0.5368229934592815
0.22522139167995192
Step 100, Loss: 0.42888924269917234
0.21832691832009005
Step 150, Loss: 0.17178116719627293
0.21733963584003504
Step 200, Loss: 0.15324474712088804
0.2172028766598669
Step 250, Loss: 0.0973674103476035
0.24602207000003545
Step 300, Loss: 0.09212524880592604
0.24245155249998787
Step 350, Loss: 0.08142786913108109
0.23846529168004055
Step 400, Loss: 0.07975307311748422
0.22228289666003548
Step 450, Loss: 0.07924188606805664
0.2419057866599178
Step 500, Loss: 0.07878873197778938


## Evaluation

After training, evaluate the model on the training set and report:
- **L2 loss** — trajectory prediction error in spectral space.
- **Residual loss** — nonlinear term consistency.
- **Nonlinear loss** — reconstruction error against $u(1 - u)$ in physical space.

In [65]:
with torch.no_grad():
    xx = torch.linspace(0, 1, num_points).to(device)
    sol = torch.tensor(sol).to(device)
    u0 = initial
    loss = 0
    loss_physical = 0 
    loss1 = 0
    net.eval()
    for step in range(num_steps):
        u = (u0 + dt * net(u0)) / (1 + dt * eigen)
        loss += torch.mean(torch.norm(u - target[:, step, :], 2, -1) / torch.norm(target[:, step, :], 2, -1))/ num_steps
        output = torch.zeros(ntrain, num_points).to(device)
        for j in range(p):
            output += u[:, j].unsqueeze(-1) * sqrt2_torch * torch.sin((j + 1) * torch.pi * xx).unsqueeze(0)
        
        loss_physical += torch.mean(torch.norm(output[:ntrain, :] - sol_torch[:ntrain, step+1, :], 2, -1) / torch.norm(sol_torch[:ntrain, step+1, :], 2, -1))/ num_steps
        Non = net(myall[:, step, :])
        ff = (myall[:, step + 1, :] - myall[:, step, :]) / dt + myall[:, step + 1, :] * eigen
        loss1 += torch.mean(torch.norm(ff - Non, 2, -1) / torch.norm(ff, 2, -1)) / num_steps
        u0 = u
    tt = torch.tensor((1. - sol) * sol).to(device)
    bb = torch.tensor(beta).to(device)
    u = net(bb)
    output = torch.zeros_like(tt)
    for j in range(p):
        output += u[:, :, j].unsqueeze(-1) * sqrt2_torch * torch.sin((j + 1) * torch.pi * xx).unsqueeze(0)
    loss2 = torch.mean(torch.norm(output[:ntrain, :num_steps, :] - tt[:ntrain, :num_steps, :], 2, -1) / torch.norm(tt[:ntrain, :num_steps, :], 2, -1))
    print("L2 loss (eigenspace): {:2.7e}, L2 loss (physical space): {:2.7e}, Residual loss: {:2.2e}, Nonlinear loss: {:2.2e}".format(loss, loss_physical, loss1, loss2))

/var/folders/xg/_l71bvm91qg1m_b2x9dlh1800000gn/T/ipykernel_12285/530140648.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sol = torch.tensor(sol).to(device)
/var/folders/xg/_l71bvm91qg1m_b2x9dlh1800000gn/T/ipykernel_12285/530140648.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tt = torch.tensor((1. - sol) * sol).to(device)


L2 loss (eigenspace): 3.0682446e-03, L2 loss (physical space): 3.0682517e-03, Residual loss: 7.57e-02, Nonlinear loss: 7.57e-02


torch.Size([100, 64])